In [1]:
import sys
import json
from tqdm import tqdm

# TO CHANGE
BASEDIR = "/home/dzigen/Desktop/PersonalAI/Personal-AI"
# TO CHNAGE

sys.path.insert(0, BASEDIR)

from src.qa_pipeline import QAPipeline, QAPipelineConfig
from src.qa_pipeline.query_parser import QueryLLMParser
from src.qa_pipeline.knowledge_comparator import KnowledgeComparator
from src.qa_pipeline.knowledge_retriever import KnowledgeRetriever, KnowledgeRetrieverConfig, AStarMetricsConfig, AStarGraphSearchConfig
from src.qa_pipeline.answer_generator import QALLMGenerator

from src.llm_agent import AgentConnector
from src.knowledge_graph_model import KnowledgeGraphModel
from src.neo4j_functions import Neo4jConnection
from src.embedding_functions import EmbeddingsDatabaseConnection, EmbeddingsDatabaseConnectionConfig, VectorDBConnectionConfig, EmbedderModelConfig

from src.embedding_functions import ChromaConnection, VectorDBConnectionConfig, EmbeddingsDatabaseConnectionConfig

In [2]:
agent = AgentConnector.open()
kg_model = KnowledgeGraphModel(
    graph_db=Neo4jConnection(uri="bolt://31.207.47.254:7687", user="neo4j", pwd="password", db_name="testdb"),
    embeddings_db=EmbeddingsDatabaseConnection(
        config=EmbeddingsDatabaseConnectionConfig(
            nodes_db_config=VectorDBConnectionConfig(
                path="../../data/graph_structures/vectorized_nodes/v8/densedb",
                db_name="vectorized_nodes"
            ),
            triplets_db_config=VectorDBConnectionConfig(
                path="../../data/graph_structures/vectorized_triplets/v4/densedb",
                db_name="vectorized_triplets"
            ),
            embedder_config=EmbedderModelConfig(
                model_name_or_path="../../models/intfloat/multilingual-e5-small"
            )
        )
    )
)

No sentence-transformers model found with name ../../models/intfloat/multilingual-e5-small. Creating a new one with MEAN pooling.


In [3]:
QUERIES = [
    "Which device is better in battery life: iPhone11 Pro Max or Xiaomi 11?", # быстро X
    "The majority of speakers have positive, neutral or negative sentiment about connection of Apple?", # ~1 min
    "The majority of speakers have positive, neutral or negative sentiment about signal of Apple?", # ~28sec
    "Kayla has positive, negative or neutral opinion about video of 10PRO on 25.11.2020?", # ~ 2 min
    "What Jessica's opinion (positive, negative or neutral) about signal of Apple was dominant during using Apple?", # ~ 25sec X
    "What opinion (positive, negative or neutral) about scheduling of IQOO9 was last during Zachary's experience of IQOO9?", # ~ 38sec
    "Do Jane and Jonathan have any common devices (which Jane and Jonathan both use)? If so, list common devices. Otherwise, answer 'No'.", # ~8sec X
    "Do Maria and Tyler prefer the same device manufacturer? If so, list common manufacturers. Otherwise, answer 'No'.", # ~42sec
    "Whose opinions from Freda and Bruce about devices are most similar to Kayla's?", # ~26sec X
    "Whose opinions from Linda and Arianna about manufacturers are most similar to Hugh's?", # 1.20min
    "Which people have positive opinion about display of Mi 10pro on 22.12.2018?", # ~ 2 min
    "Which people have negative opinion about video of 10PRO on 25.11.2020?" # ~ 2 min
    ]

### QA-пайплан целиком

In [4]:
qa_pipeline = QAPipeline(kg_model, agent, config=QAPipelineConfig(knowledge_retriever_config=KnowledgeRetrieverConfig(
    graph_retriever_config=AStarGraphSearchConfig(metrics_config=AStarMetricsConfig(
        nodes_distances_path = '../../data/graph_structures/vectorized_nodes/v8/nodes_distances_matrix',
        nodes_short_paths_file = '../../data/graph_structures/graph_short_paths/stage2/v2/distances_matrix'
    )))))
#qa_pipeline.answer(QUERY)

In [ ]:
qa_pipeline.answer(QUERIES[6])

In [5]:
import os
EVAL_DATADIR = '../../data/qa_eval'
qa_files = os.listdir(EVAL_DATADIR)

In [16]:
for qa_file in qa_files[10:11]:
    print(qa_file)
    with open(f"{EVAL_DATADIR}/{qa_file}", 'r', encoding='utf-8') as fd:
        data = json.loads(fd.read())

    gen_answers = []
    process = tqdm(data)
    for qa_pair in process:
        gen_answer = qa_pipeline.answer(qa_pair['question'])
        gen_answers.append({"generated_answer": gen_answer})
        process.set_postfix({'target': qa_pair['answer'], 'generated': gen_answer})
    
    with open(f"./logs/generated/{qa_file}", 'w', encoding='utf-8') as fd:
        fd.write(json.dumps(gen_answers, indent=1, ensure_ascii=False))

dominant_opinion.json


  3%|▎         | 26/902 [14:02<7:55:51, 32.59s/it, target=negative, generated=Negative]               

### QA-пайплайн по частям

In [40]:
# stage 1
q_parser = QueryLLMParser(agent)
qparser_out = q_parser.extract_entities(QUERIES[11])
print(qparser_out.entities)

['people', 'opinion', 'video', '10PRO', '25.11.2020']


In [41]:
# stage 2
k_comparator = KnowledgeComparator(kg_model)
k_comparator.link_kgnodes_to_query(qparser_out)

for item in qparser_out.linked_nodes:
    print(item)

VectorDBInstance(id='4:d958299b-8cff-4454-876f-4f337d0518bd:1389', document='leonars (kind: person)', embedding=[0.05517178773880005, 0.002405497943982482, -0.04323551431298256, -0.073570616543293, 0.04137517511844635, -0.05307059362530708, 0.02288951352238655, 0.02365204691886902, 0.017923245206475258, 0.03855716437101364, 0.04861706122756004, -0.011322339065372944, 0.039754077792167664, -0.03724907338619232, -0.07034523785114288, 0.012743735685944557, 0.032692842185497284, -0.04549724981188774, -0.035445231944322586, -0.042756807059049606, 0.029045304283499718, 0.013236959464848042, -0.04126825928688049, 0.04899909347295761, 0.02234731987118721, 0.06407668441534042, -0.05949537828564644, 0.009342987090349197, 0.07674012333154678, -0.04076053947210312, -0.06230504438281059, 0.0021265309769660234, 0.0657259151339531, 0.00030344558763317764, 0.07424210011959076, 0.04435907304286957, -0.0589015930891037, -0.05719950050115585, 0.0193796344101429, -0.031495966017246246, 0.00549998274073004

In [42]:
# stage 3
k_retriever = KnowledgeRetriever(kg_model, config=KnowledgeRetrieverConfig(
    graph_retriever_config=AStarGraphSearchConfig(metrics_config=AStarMetricsConfig(
        nodes_distances_path = '../../data/graph_structures/vectorized_nodes/v8/nodes_distances_matrix',
        nodes_short_paths_file = '../../data/graph_structures/graph_short_paths/stage2/v2/distances_matrix'
    ))))
kretriever_out = k_retriever.retrieve(qparser_out)

for item in kretriever_out:
    print(item)

Количество извлечённых триплетов: 36
Количество триплетов после фильтрации: 36
Triplet(start_node=Node(name='freda', type=<NodeType.object: 'object'>, id='4:d958299b-8cff-4454-876f-4f337d0518bd:1187', prop={'kind': 'person', 'name': 'freda'}, stringified='freda (kind: person)'), relation=Relation(name='has_device', type=<RelationType.simple: 'simple'>, id='5:d958299b-8cff-4454-876f-4f337d0518bd:7771', prop={'name': 'has_device'}), end_node=Node(name='s21_ultra', type=<NodeType.object: 'object'>, id='4:d958299b-8cff-4454-876f-4f337d0518bd:665', prop={'kind': 'device', 'name': 's21_ultra'}, stringified='s21_ultra (kind: device)'), id='147ec36b6ddda67ab811314eeef72f95', stringified=None)
Triplet(start_node=Node(name='12p', type=<NodeType.object: 'object'>, id='4:d958299b-8cff-4454-876f-4f337d0518bd:38', prop={'kind': 'device', 'name': '12p'}, stringified='12p (kind: device)'), relation=Relation(name='opinion', type=<RelationType.simple: 'simple'>, id='5:d958299b-8cff-4454-876f-4f337d0518b

In [43]:
# stage 4
qa_generator = QALLMGenerator(agent)
contexts = qa_generator.formate_context(kretriever_out)
qagenerator_out = qa_generator.generate(qparser_out.query, contexts)

print(contexts)
print()
print(qagenerator_out)

- freda (kind: person) has_device s21_ultra (kind: device)
- 28.11.2020: 12p (kind: device) opinion (sentiment: pos; person: Matthew; opinion: amazing_its_clear) resolution (kind: feature)
- 3.5.2020: 12p (kind: device) opinion (sentiment: neg; person: Riley; opinion: a_little_worse) resolution (kind: feature)
- 21.12.2020: 11pro (kind: device) opinion (sentiment: neu; person: Ashley; opinion: not_blown_away) battery_life (kind: feature)
- 7.6.2020: 11pro (kind: device) opinion (sentiment: neu; person: Deborah; opinion: its_adequate) battery_life (kind: feature)
- 19.12.2020: 11pro (kind: device) opinion (sentiment: neu; person: Danielle; opinion: its_decent_though) battery_life (kind: feature)
- 2.7.2020: 11pro (kind: device) opinion (sentiment: pos; person: Audrey; opinion: lasts_all_day) battery_life (kind: feature)
- 12.9.2020: 11pro (kind: device) opinion (sentiment: pos; person: Morgan; opinion: its_been_lifesaver) battery_life (kind: feature)
- 1.11.2020: 11pro (kind: device) op